In [ ]:
import os
import locale
import modal
from agents.preprocessor import Preprocessor
from dotenv import load_dotenv
load_dotenv(override=True)

In [ ]:
print(locale.getpreferredencoding())

In [ ]:
os.environ["PYTHONIOENCODING"] = "utf-8"

In [ ]:
preprocessor = Preprocessor()
text = preprocessor.preprocess("Quadcast HyperX condenser mic, connects via usb-c to your computer for crystal clear audio")
print(text)

In [ ]:
!uv run modal deploy -m pricer_service

In [ ]:
Pricer = modal.Cls.from_name("pricer-service", "Pricer")
pricer = Pricer()
reply = pricer.price.remote(text)
print(reply)

In [ ]:
reply = pricer.price.remote(text)
print(reply)

In [ ]:
from agents.specialist_agent import SpecialistAgent

In [ ]:
agent = SpecialistAgent()

In [ ]:
agent.price("iPhone 16")

In [ ]:
from dotenv import load_dotenv
import os
from huggingface_hub import Collection, login
from openai import OpenAI
import chromadb
from tqdm import tqdm
from agents.items import Item

In [ ]:
# Load the environment variables
load_dotenv(override=True)

# Global variables
VECTOR_DB = "products_vectorstore"

In [ ]:
db_client = chromadb.PersistentClient(path=VECTOR_DB)

In [ ]:
collection_name = "products"
existing_collection_names = [collection.name for collection in db_client.list_collections()]

In [ ]:
existing_collection_names

In [ ]:
if collection_name in existing_collection_names:
    db_client.delete_collection(name=collection_name)

In [ ]:
from openai import OpenAI

In [ ]:
EMBEDDING_MODEL = "text-embedding-3-small"


In [ ]:
openai = OpenAI()
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Hi there!"}
]
resposne = openai.chat.completions.create(model="gpt-5-nano", messages=messages)
print(resposne.choices[0].message.content)

In [ ]:
openai = OpenAI()
documents = ["documents to be embedded", "Another doucment which also needs to be embedded"]
responses = openai.embeddings.create(input=documents, model=EMBEDDING_MODEL)
embeddings = [item.embedding for item in responses.data]

In [ ]:
embeddings

In [ ]:
from dotenv import load_dotenv
import os
from huggingface_hub import Collection, login
from openai import OpenAI
import chromadb
from tqdm import tqdm
from agents.items import Item

In [ ]:
# Load the environment variables
load_dotenv(override=True)

# Global variables
VECTOR_DB = "products_vectorstore"
LITE_MODE = True
EMBEDDING_MODEL = "text-embedding-3-small"

In [ ]:
def get_data_from_huggingface() -> list[Item]:
    """ Login into huggingface and get data which will be loaded in vector database """

    # Login to huggingface
    print("Logging into Hugging Face...")
    hf_token = os.getenv("HF_TOKEN")
    login(token=hf_token, add_to_git_credential=False)

    print("Logged in. Fetching dataset...")

    # Get data from huggingface
    username = "ed-donner"
    dataset_name = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"
    train, val, test = Item.from_hub(dataset_name=dataset_name)

    print(f"Downloaded {len(train)} data from Hugging face.")

    return train

In [ ]:
db_client = chromadb.PersistentClient(path=VECTOR_DB)

In [ ]:
print(len(train))

In [ ]:
openai = OpenAI()

In [ ]:
# Check if the collection exists, if not create it
collection_name = "products"
existing_collection_names = [collection.name for collection in db_client.list_collections()]
existing_collection_names

In [ ]:
if collection_name in existing_collection_names:
    db_client.delete_collection(name=collection_name)

In [ ]:
db_client.delete_collection(name=collection_name)
collection = db_client.create_collection(name=collection_name)
for i in tqdm(range(0, len(train), 1000)):
    documents = [item.summary for item in train[i: i+1000]]
    response = openai.embeddings.create(input=documents, model=EMBEDDING_MODEL)
    embeddings = [item.embedding for item in response.data]
    print(embeddings)
    metadatas = [{"category": item.category, "price": item.price} for item in train[i: i+1000]]
    ids = [f"doc_{j}" for j in range(i, i+1000)]
    ids = ids[:len(documents)]
    collection.add(ids=ids, embeddings=embeddings, metadatas=metadatas, documents=documents)

In [8]:
from agents.frontier_agent import FrontierAgent
import chromadb
from dotenv import load_dotenv
from openai import OpenAI

In [9]:
# Load the environment variables
load_dotenv(override=True)

# Global variables
VECTOR_DB = "products_vectorstore"
LITE_MODE = True
EMBEDDING_MODEL = "text-embedding-3-small"
client = OpenAI()

In [10]:
collection_name = "products"
db_client = chromadb.PersistentClient(path=VECTOR_DB)
collection = db_client.get_or_create_collection(collection_name)

INFO:backoff:Backing off send_request(...) for 0.3s (requests.exceptions.ReadTimeout: HTTPSConnectionPool(host='us.i.posthog.com', port=443): Read timed out. (read timeout=15))


In [ ]:
description = "Old Blood Noise Excess V2 Distortion Chorus/Delay Pedal"

In [ ]:
frontier_agent = FrontierAgent(collection=collection)
result = frontier_agent.price(description=description)
print(result)

In [ ]:
from agents.specialist_agent import SpecialistAgent

In [ ]:
specialist_agent = SpecialistAgent()
result = specialist_agent.price(description=description)
print(result)

In [ ]:
from agents.neural_network_agent import NeuralNetworkAgent

In [ ]:
network_agent_agent = NeuralNetworkAgent()
result = network_agent_agent.price(description=description)
print(result)

In [ ]:
from agents.ensemble_agent import EnsembleAgent

In [ ]:
ensemble_agent = EnsembleAgent(collection=collection)
result = ensemble_agent.price(description=description)
print(result)

In [ ]:
from agents.scanner_agent import ScannerAgent

In [ ]:
scanner_agent = ScannerAgent()
result = scanner_agent.scan()
print(result)

In [11]:
import logging

In [12]:
root = logging.getLogger()
root.setLevel(level=logging.INFO)

In [ ]:
from agents.autonomous_planning_agent import AutonomousPlanningAgent
agent = AutonomousPlanningAgent(collection=collection)

In [ ]:
agent.plan()

In [13]:
from agents.planning_agent import PlanningAgent
agent = PlanningAgent(collection)

INFO:root:[Planning Agent] Planning Agent is initializing
INFO:root:[Scanner Agent] Scanner Agent is initializing
INFO:root:[Scanner Agent] Scanner Agent is ready
INFO:root:[Ensemble Agent] Initializing Ensemble Agent
INFO:root:[Frontier Agent] Initializing Frontier Agent
INFO:root:[Frontier Agent] Frontier Agent is setting up with OpenAI
INFO:root:[Frontier Agent] Frontier Agent is ready
INFO:root:[Specialist Agent] Specialist Agent is initializing - connecting to modal
INFO:root:[Neural Network Agent] Neural Network Agent is initializing
INFO:root:Neural Network is using cpu
INFO:root:[Neural Network Agent] Neural Network Agent is ready and weights are loaded
INFO:root:[Ensemble Agent] Ensemble Agent is ready
INFO:root:[Messaging Agent] Messaging Agent is initializing
INFO:root:[Messaging Agent] Messaging Agent has initialized Pushover and Claude
INFO:root:[Planning Agent] Planning Agent is ready


In [14]:
agent.plan()

INFO:root:[Planning Agent] Planning Agent is kicking off a run
INFO:root:[Scanner Agent] Scanner Agent is about to fetch deals from RSS feed
INFO:root:[Scanner Agent] Scanner Agent got 30 deals not already scraped
INFO:root:[Scanner Agent] Scanner Agent is calling OpenAI using structured output
INFO:root:[Scanner Agent] Scanner Agent got 5 selected deals with price > 0 from OpenAI
INFO:root:[Planning Agent] Planning Agent is pricing a potential deal
INFO:root:[Ensemble Agent] Running Ensemble Agent - processing text
01:44:01 - LiteLLM:INFO: utils.py:3898 - 
LiteLLM completion() model= openai/gpt-oss-20b; provider = groq
INFO:LiteLLM:
LiteLLM completion() model= openai/gpt-oss-20b; provider = groq
01:44:02 - LiteLLM:INFO: utils.py:1630 - Wrapper: Completed Call, calling success_handler
INFO:LiteLLM:Wrapper: Completed Call, calling success_handler
INFO:root:[Ensemble Agent] Preprocessed text using groq/openai/gpt-oss-20b
INFO:root:[Frontier Agent] Frontier Agent is performing a RAG searc

Opportunity(deal=Deal(product_description='Anker Solix F3000 Portable Power Station is a high-capacity unit rated for 3,600W output that supports pass-through charging and ultra-low idle draw for extended standby. It offers dual solar inputs supporting up to 2,400W charging, expandable capacity up to 24kWh, and the ability to pair two units for 240V high-demand appliance support; intended for heavy-duty backup power and off-grid use.', price=1234.0, url='https://www.dealnews.com/products/Anker-Solix/Anker-Solix-F3000-3-600-W-Portable-Power-Station/497959.html?iref=rss-c142'), estimate=2473.9891220092777, discount=1239.9891220092777)